# Generación de Batería de Preguntas para TFG

Este notebook tiene dos partes:
1. **Exploración de la Base de Datos:** Usamos Elasticsearch con parámetros flexibles para descubrir de qué hablan las noticias indexadas y extraer hechos concretos.
2. **Generación del Dataset:** Creación y guardado del archivo `bateria_preguntas_tfg.csv` con el formato requerido por el Juez LLM.

In [1]:
from elasticsearch import Elasticsearch

# 1. CONEXIÓN (Túnel SSH)
es = Elasticsearch("http://127.0.0.1:9250")

try:
    if es.ping():
        print(f"Conectado a: {es.info()['name']} (Listo para explorar)")
    else:
        print("El servidor no responde. ¿Está el túnel abierto?")
except Exception as e:
    print(f"Error inesperado: {e}")

# 2. FUNCIÓN DE EXPLORACIÓN (Relajada)
def explorar_noticias(query, k=4): # k=3 para no saturar la pantalla
    print(f"\nEXPLORANDO TEMA: '{query}'")
    print("=" * 80)
    
    search_payload = {
        "size": k,
        "query": {
            "multi_match": {
                "query": query,
                "fields": ["title^3", "body"], 
                "fuzziness": "AUTO", # Tolerancia a errores (solo para explorar)
                "operator": "or"     # Búsqueda amplia
            }
        },
        # IMPORTANTE: Traemos el body para poder leer el dato exacto
        "_source": ["title", "date", "source", "body"] 
    }
    
    try:
        response = es.search(index="noticias_tfg", body=search_payload) 
        hits = response['hits']['hits']
        
        if not hits:
            print("No se encontraron noticias sobre este tema.")
            return

        for i, hit in enumerate(hits):
            data = hit['_source'] 
            clean_date = data.get('date', 'N/A')[:10] if data.get('date') else "Sin fecha"
            
            print(f"{i+1}. TÍTULO: {data.get('title', 'Sin título')}")
            print(f"FECHA:  {clean_date} | FUENTE: {data.get('source', 'Desconocida')}")
            # Imprimimos los primeros 400 caracteres del cuerpo para buscar el "dato"
            cuerpo = data.get('body', '')
            print(f"TEXTO:  {cuerpo}...\n")
            print("-" * 80)
            
    except Exception as e:
        print(f"Error durante la búsqueda: {e}")

Conectado a: valencia (Listo para explorar)


In [2]:
explorar_noticias("telescopio James Webb descubrimiento exoplaneta")

explorar_noticias("elecciones El Salvador Bukele resultados")

explorar_noticias("tratamiento CRISPR anemia celulas falciformes")

explorar_noticias("tunel submarino Fehmarnbelt Dinamarca Alemania")

explorar_noticias("cumbre del clima COP29 Baku acuerdos")

explorar_noticias("aprobacion vacuna malaria R21 OMS")


EXPLORANDO TEMA: 'telescopio James Webb descubrimiento exoplaneta'
1. TÍTULO: El telescopio James Webb detecta posibles huellas químicas de vida en el exoplaneta K2-18 b
FECHA:  2025-04-17 | FUENTE: La Razón
TEXTO:  El telescopio espacial James Webb ha identificado en la atmósfera del exoplaneta K2-18 b compuestos como el dimetil sulfuro (DMS) y el dimetil disulfuro (DMDS), que en la Tierra son producidos por organismos vivos, especialmente fitoplancton marino. Este hallazgo, aunque no constituye una prueba definitiva de vida extraterrestre, representa la evidencia más sólida hasta la fecha de posibles signos biológicos más allá de nuestro sistema solar....

--------------------------------------------------------------------------------
2. TÍTULO: El descubrimiento del telescopio James Webb de una cadena de 20 galaxias de 13 millones de años luz de largo
FECHA:  2025-09-19 | FUENTE: Desconocido
TEXTO:  La ciencia se encuentra en constante avance y los hallazgos de los astrónomos no d

In [1]:
import pandas as pd

# Definimos la batería de preguntas manualmente
datos = [
    {
        "id": 1,
        "tipo": "Factual",
        "pregunta": "¿A cuanto llego a subir Trump los aranceles a China?",
        "respuesta_esperada": "145%."
    },
    {
        "id": 2,
        "tipo": "Factual",
        "pregunta": "¿Cuánto dinero perdió Elon Musk tras el anuncio de los aranceles?",
        "respuesta_esperada": "17.800 millones de dólares."
    },
    
    {
        "id": 3,
        "tipo": "Trampa",
        "pregunta": "¿Qué opinó el Rey de España sobre los aranceles de Trump a China?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
     {
        "id": 4,
        "tipo": "Trampa",
        "pregunta": "¿Qué declaraciones hizo Elon Musk sobre los aranceles de donald Trump a China?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    {
        "id": 5,
        "tipo": "Inferencia",
        "pregunta": "¿Qué relación quiere Elon Musk entre Estados Unidos y Europa según sus declaraciones en Italia?",
        "respuesta_esperada": "Quiere una zona de libre comercio con aranceles cero."
    },

    {
        "id": 6,
        "tipo": "Factual",
        "pregunta": "¿Qué presidente firmó un decreto para centralizar la regulación de la inteligencia artificial?",
        "respuesta_esperada": "Donald Trump."
    },

    {
        "id": 7,
        "tipo": "Inferencia",
        "pregunta": "¿Por qué Donald Trump quiere quitar a los estados el derecho a regular la inteligencia artificial por su cuenta?",
        "respuesta_esperada": "Debe haber un único manual si queremos seguir liderando en inteligencia artificial"
    },

    {
        "id": 8,
        "tipo": "Trampa",
        "pregunta": "¿Qué opinó Felipe González sobre el decreto de inteligencia artificial de Trump?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    {
        "id": 9,
        "tipo": "Factual",
        "pregunta": "¿Por cuántas temporadas ha fichado Kylian Mbappé con el Real Madrid?",
        "respuesta_esperada": "Por cinco temporadas."
    },

    {
        "id": 10,
        "tipo": "Inferencia",
        "pregunta": "¿Por qué el artículo de El País dice que el comunicado del fichaje de Mbappé cayó en medio de una resaca emocional?",
        "respuesta_esperada": "Porque el anuncio se hizo menos de 24 horas después de que terminaran las celebraciones por la 15ª Copa de Europa del Real Madrid."
    },

    {
        "id": 11,
        "tipo": "Factual",
        "pregunta": "¿Cuál dijo Pedro Sánchez que era su principal prioridad en la clausura del 41 Congreso Federal del PSOE?",
        "respuesta_esperada": "ganar las elecciones municipales y autonómicas de 2027 y volver a gobernar en toda España"
    },

    {
        "id": 12,
        "tipo": "Trampa", 
        "pregunta": "¿Cuando dimitió Pedro Sánchez como presidente?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    {
        "id": 13,
        "tipo": "Factual",
        "pregunta": "¿Cuanto patrimonio tiene Elon Musk?",
        "respuesta_esperada": "Mas de 400.000 millones de dólares"
    },
    {
        "id": 14,
        "tipo": "Inferencia",
        "pregunta": "¿Por qué fue demandado Elon Musk por la Fiscalía de Filadelfia en 2024?",
        "respuesta_esperada": "Porque prometió dar 1 millón de dolares a votantes de las elecciones"
    },
    {
        "id": 15,
        "tipo": "Trampa", 
        "pregunta": "¿Cuántos bitcoins propuso repartir Donald Trump como renta básica universal?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 16,
        "tipo": "Factual", 
        "pregunta": "¿Cuántas personas murieron en la dana de Valencia?",
        "respuesta_esperada": "227 personas."
    },
    # --- GEOPOLÍTICA Y SOCIEDAD (Guerra en Ucrania) ---
    {
        "id": 17,
        "tipo": "Factual",
        "pregunta": "¿Qué enfermedades se asocian al uranio de los proyectiles antitanque utilizados en la guerra en Ucrania?",
        "respuesta_esperada": "Algunos estudios lo asocian con cáncer, problemas renales o efectos en el desarrollo infantil."
    },
    {
        "id": 18,
        "tipo": "Inferencia",
        "pregunta": "¿Por qué el aumento de polacos y ucranianos que compran vivienda en España para huir de la guerra apenas recurre a la financiación de una hipoteca?",
        "respuesta_esperada": "Porque tienen un poder adquisitivo elevado y un patrimonio alto que les permite no necesitar una hipoteca."
    },
    {
        "id": 19,
        "tipo": "Factual",
        "pregunta": "¿A quién designó Donald Trump para servir como secretario de Defensa en 2025?",
        "respuesta_esperada": " Donald Trump designó a Pete Hegseth."
    },
    {
        "id": 20,
        "tipo": "Trampa",
        "pregunta": "¿Cuántos soldados estadounidenses ha prometido enviar Donald Trump para formar una fuerza de paz en Ucrania?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- DANA de Valencia ---
    {
        "id": 21,
        "tipo": "Factual",
        "pregunta": "¿Qué cantidad económica donó el Grupo Renault a los afectados por la DANA en Valencia y quién es el CEO de la compañía?",
        "respuesta_esperada": "Donó un millón de euros y su CEO es Luca de Meo."
    },
    {
        "id": 22,
        "tipo": "Inferencia",
        "pregunta": "Durante su concierto en Valencia, ¿qué canción improvisó Bryan Adams y con qué propósito en relación a la DANA?",
        "respuesta_esperada": "Improvisó una versión de 'Come Together' de The Beatles como símbolo de unión entre el artista y la comunidad afectada."
    },
    {
        "id": 23,
        "tipo": "Trampa",
        "pregunta": "¿Qué cantidad económica donó el futbolista Kylian Mbappé a los afectados por la DANA de Valencia?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 24,
        "tipo": "Inferencia",
        "pregunta": "¿A quien iba dirigido el mensaje en el tapiz floral de la Basílica durante la festividad de la patrona de Valencia tras el paso de la DANA?",
        "respuesta_esperada": "a los voluntarios que acudieron a las zonas afectadas para ayudar de manera desinteresada tras la dana"
    },
    {
        "id": 25,
        "tipo": "Factual",
        "pregunta": "¿Con cuántas psicólogas especialistas se reforzó la atención del teléfono 016 en los municipios afectados por la DANA en Valencia?",
        "respuesta_esperada": "Con cuatro psicólogas especialistas."
    },
    # --- ESPACIO Y TECNOLOGÍA (Misiones Lunares) ---
    {
        "id": 26,
        "tipo": "Factual",
        "pregunta": "Ante el retraso de SpaceX en la misión a la Luna y la actual guerra en la NASA por su jefatura, ¿a qué otra empresa mencionó el administrador interino Sean Duffy para abrir los contratos del módulo de aterrizaje?",
        "respuesta_esperada": "Mencionó a Blue Origin, la empresa de Jeff Bezos."
    },
    {
        "id": 27,
        "tipo": "Inferencia",
        "pregunta": "¿Qué fallo provocó que la nave Hakuto-R de SpaceX se estrellara en su intento anterior en 2023?",
        "respuesta_esperada": "Se estrelló porque calculó mal su descenso al pasar sobre un cráter, debido a datos imprecisos sobre la altitud y problemas de software."
    },
    {
        "id": 28,
        "tipo": "Trampa",
        "pregunta": "¿Qué astronauta lidera la tripulación del módulo Blue Ghost en las nuevas misiones a la Luna de SpaceX?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos (el texto especifica que es una misión no tripulada)."
    },

    # --- SALUD Y CIENCIA (Vacunas y Cáncer) ---
    {
        "id": 29,
        "tipo": "Inferencia",
        "pregunta": "¿Por qué la nueva vacuna de ARN para el tratamiento del cáncer se tiene que elaborar de manera personalizada para cada paciente?",
        "respuesta_esperada": "Porque los neoantígenos son proteínas anómalas que resultan de mutaciones genéticas, y el perfil de mutaciones es distinto y único en el tumor de cada paciente."
    },
    {
        "id": 30,
        "tipo": "Trampa",
        "pregunta": "¿Qué tipo de cáncer específico le fue diagnosticado al rey Carlos III?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- DEPORTES (Carlos Alcaraz) ---
    {
        "id": 31,
        "tipo": "Factual",
        "pregunta": "¿Cuánto dinero aproximado le quedó limpio a Carlos Alcaraz por ganar su sexto Grand Slam tras pagar los impuestos federales y de Nueva York?",
        "respuesta_esperada": "Le quedan aproximadamente 2,7 millones de dólares."
    },
    {
        "id": 32,
        "tipo": "Inferencia",
        "pregunta": "¿Qué golpe fundamental ha modificado Carlos Alcaraz en su juego para el Open de Australia??",
        "respuesta_esperada": "Ha modificado su saque (el servicio)."
    },

    # --- ECONOMÍA Y VIVIENDA ---
    {
        "id": 33,
        "tipo": "Factual",
        "pregunta": "Mientras el precio de la vivienda frenaba su subida en verano, ¿En que dos comunidades bajó el precio?",
        "respuesta_esperada": "En Castilla-La Mancha (caída del 12,8%) y Asturias (caída del 3,1%)."
    },
    {
        "id": 34,
        "tipo": "Inferencia",
        "pregunta": "¿Qué medida propone Santiago Niño Becerra para evitar la constante revalorización del mercado y poner solución a los problemas de vivienda en España?",
        "respuesta_esperada": "Propone crear un gran parque de vivienda pública destinado exclusivamente al alquiler."
    },

    # --- GEOPOLÍTICA TECNOLÓGICA (TikTok en EE.UU.) ---
    {
        "id": 35,
        "tipo": "Factual",
        "pregunta": "¿A partir de que día se prohibe Tiktok en Estados Unidos?",
        "respuesta_esperada": "A partir del domingo 19 de enero de 2025."
    },
    # --- DEPORTES (Eurocopa y Fútbol) ---
    {
        "id": 36,
        "tipo": "Factual",
        "pregunta": "¿Qué jugadores metieron gol para asegurar el 2-1 en el España-Inglaterra, dándole a España su cuarta Eurocopa?",
        "respuesta_esperada": "Nico Williams y Mikel Oyarzabal."
    },
    {
        "id": 37,
        "tipo": "Inferencia",
        "pregunta": "¿Qué dos hechos deportivos que sucedieron el mismo dia hicieron que L'Équipe titulara 'los nuevos reyes de España'?",
        "respuesta_esperada": "Porque coincidió que en el mismo día los futbolistas de la selección se proclamaron campeones de Europa y el tenista Carlos Alcaraz retuvo su título en Wimbledon."
    },
    {
        "id": 38,
        "tipo": "Factual",
        "pregunta": "¿Cuántos autogoles o goles en propia puerta se le atribuyen a Harry Kane en la final de la Eurocopa 2024",
        "respuesta_esperada": "Ninguno"
    },
    {
        "id": 39,
        "tipo": "Trampa",
        "pregunta": "Tras el partido en el que la España de las finales vuelve a ganar la Liga de Naciones, ¿qué entrenadora alemana fue destituida de su cargo?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- ECONOMÍA Y CONSUMO (Aceite de Oliva) ---
    {
        "id": 40,
        "tipo": "Factual",
        "pregunta": "Tras el giro del precio del aceite de oliva, ¿a cuánto vende el supermercado Alcampo su garrafa de 5 litros de la marca Suroliva?",
        "respuesta_esperada": "A 30,90 euros."
    },
    {
        "id": 41,
        "tipo": "Inferencia",
        "pregunta": "¿Cuáles son las razones por las que la garrafa de 5 litros de aceite Suroliva ha bajado a 30,90 euros?",
        "respuesta_esperada": "debido a una combinación de factores que incluyen la necesidad de liberar stock, la anticipación de una buena cosecha que estabilizará la oferta y la competencia entre supermercados para atraer a los consumidores."
    },
    {
        "id": 42,
        "tipo": "Trampa",
        "pregunta": "¿Qué cadena de supermercados está regalando garrafas de aceite de oliva en Madrid para compensar los problemas de la sequía?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- CULTURA Y SOCIEDAD (Taylor Swift) ---
    {
        "id": 43,
        "tipo": "Factual",
        "pregunta": "¿De cuántos millones ha sido el impacto económico en Madrid por la hostelería gracias a los conciertos de Taylor Swift?",
        "respuesta_esperada": "10 millones de euros."
    },
    {
        "id": 44,
        "tipo": "Inferencia",
        "pregunta": "Durante su enorme impacto económico en la gira musical, ¿por qué Taylor Swift decidió interpretar las reediciones denominadas 'Taylor's Version' de sus antiguos álbumes?",
        "respuesta_esperada": "Para recuperar la propiedad intelectual de su discografía después de que su discográfica anterior le quitara los derechos de sus primeros 7 albumes."
    },
    {
        "id": 45,
        "tipo": "Trampa",
        "pregunta": "Para los conciertos de Taylor Swift en Madrid, ¿qué famoso jugador de baloncesto subió al escenario del Santiago Bernabéu como invitado sorpresa?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 46,
        "tipo": "Trampa",
        "pregunta": "¿Cuánto dinero le cobró exactamente el presidente del Real Madrid a Taylor Swift por alquilar el estadio de fútbol?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- DEPORTES GLOBALES (Juegos Olímpicos) ---
    {
        "id": 47,
        "tipo": "Factual",
        "pregunta": "En relación con el calendario de los Juegos Olímpicos de París 2024, ¿en qué lugar exacto se celebrarán las competiciones de surf para aprovechar las olas?",
        "respuesta_esperada": "En Teahupo’o, Tahití."
    },
    {
        "id": 48,
        "tipo": "Trampa",
        "pregunta": "¿Qué material se utilizó específicamente para las palas de padel de los deportistas españoles en los Juegos de París 2024?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- POLÍTICA INTERNACIONAL (Francia y Argentina) ---
    {
        "id": 49,
        "tipo": "Inferencia",
        "pregunta": "¿Qué decisión anunció Emmanuel Macron tras la victoria de Le Pen?",
        "respuesta_esperada": "Disolver de la Asamblea Nacional y convocar elecciones legislativas"
    },
    {
        "id": 50,
        "tipo": "Factual",
        "pregunta": "En los primeros cien días de su mandato para combatir la crisis, ¿a qué cifra de inflación anual se enfrentó Javier Milei en Argentina al cierre del año 2023?",
        "respuesta_esperada": "A una inflación anual del 211,4%."
    },
    {
        "id": 51,
        "tipo": "Inferencia",
        "pregunta": "Según la situación económica que vive la Argentina de Javier Milei con su alta inflación, ¿por qué motivo exacto una jueza de Nueva York rechazó al país la extensión del plazo para pagar la deuda de YPF?",
        "respuesta_esperada": "Porque consideró que el Gobierno argentino no había tomado medidas reales para afrontar el pago de la indemnización ni tenía un cronograma establecido para hacerlo."
    },
    {
        "id": 52,
        "tipo": "Trampa",
        "pregunta": "Tras convocar elecciones legislativas en Francia, ¿qué cargo público le ofreció Emmanuel Macron a Marine Le Pen para evitar dimitir?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 53,
        "tipo": "Trampa",
        "pregunta": "¿Con qué porcentaje de votos ganó el partido de Los Republicanos las elecciones presidenciales de Francia en 2024?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 54,
        "tipo": "Trampa",
        "pregunta": "¿A qué jugador de la selección nacional de Francia despidió personalmente Javier Milei en Argentina tras el incidente con los cantos racistas?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 55,
        "tipo": "Trampa",
        "pregunta": "¿Cuántos millones de dólares confesó haber robado Javier Milei en la estafa del llamado Criptogate en Argentina?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 56,
        "tipo": "Factual",
        "pregunta": "¿Qué actor protagoniza la película 'On the Waterfront' (1953) mencionada en el artículo de los Óscar?",
        "respuesta_esperada": "Marlon Brando"
    },
    {
        "id": 57,
        "tipo": "Factual",
        "pregunta": "¿Qué sector profesional organizó las protestas conocidas como 'tractoradas' en distintos puntos de España?",
        "respuesta_esperada": "Los agricultores."
    },
    {
        "id": 58,
        "tipo": "Factual",
        "pregunta": "Según los reportes de RTVE de febrero de 2025, ¿con qué objetivo principal retomaron las tractoradas los agricultores en diferentes puntos de España?",
        "respuesta_esperada": "Para pedir cambios en el sector y exigir derechos."
    },
    {
        "id": 59,
        "tipo": "Factual",
        "pregunta": "¿En qué mes del año 2024 organizaron cientos de agricultores tractoradas con la intención de bloquear España?",
        "respuesta_esperada": "En el mes de febrero."
    },
    {
        "id": 60,
        "tipo": "Factual",
        "pregunta": "¿Qué tipo de vehículos utilizaron los agricultores en sus manifestaciones para intentar bloquear el país?",
        "respuesta_esperada": "Tractores."
    },
    {
        "id": 61,
        "tipo": "Inferencia",
        "pregunta": "¿qué paso en España con cientos de agricultores el martes 6 de febrero de 2024?",
        "respuesta_esperada": "El martes 6 de febrero de 2024, cientos de agricultores en España se movilizaron en protesta"
    },
    {
        "id": 62,
        "tipo": "Factual",
        "pregunta": "¿Qué película ganó el premio a Mejor Película en la edición de los premios Oscar de 2025?",
        "respuesta_esperada": "Anora."
    },
    {
        "id": 63,
        "tipo": "Factual",
        "pregunta": "¿Qué cineasta ganó el premio Oscar a Mejor Director en 2025 por la película 'Anora'?",
        "respuesta_esperada": "Sean Baker."
    },
    {
        "id": 64,
        "tipo": "Factual",
        "pregunta": "¿Qué actriz se llevó el Oscar a Mejor Actriz Protagonista por su papel en la película 'Anora'?",
        "respuesta_esperada": "Mikey Madison."
    },
    {
        "id": 65,
        "tipo": "Factual",
        "pregunta": "¿Qué actor logró el premio Oscar a Mejor Actor Protagonista en 2025 por su actuación en la película 'The Brutalist'?",
        "respuesta_esperada": "Adrien Brody ganó el premio Óscar a Mejor Actor Protagonista en 2025."
    },
    {
        "id": 66,
        "tipo": "Factual",
        "pregunta": "¿Cuál fue la película que ganó el Oscar a Mejor Película Internacional en 2025?",
        "respuesta_esperada": "I'm Still Here (Aún estoy aquí) de Brasil."
    },
    {
        "id": 67,
        "tipo": "Factual",
        "pregunta": "¿Qué partido político sufrió una derrota histórica en las elecciones del Reino Unido en julio de 2024?",
        "respuesta_esperada": "El Partido Conservador."
    },
    {
        "id": 68,
        "tipo": "Factual",
        "pregunta": "Tras los resultados de las elecciones en el Reino Unido de 2024, ¿qué partido político formó el nuevo Gobierno",
        "respuesta_esperada": "El Partido Laborista (Labour Party)."
    },
    {
        "id": 69,
        "tipo": "Factual",
        "pregunta": "¿En qué país europeo se celebraron elecciones a principios de julio de 2024 en las que los conservadores sufrieron una derrota histórica?",
        "respuesta_esperada": "En el Reino Unido."
    },
    {
        "id": 70,
        "tipo": "Factual",
        "pregunta": "¿Qué política se convirtió en la primera mujer en encabezar el Ministerio de Economía en la historia del Reino Unido al formarse el nuevo Gobierno en 2024?",
        "respuesta_esperada": "Rachel Reeves."
    },
    {
        "id": 71,
        "tipo": "Factual",
        "pregunta": "¿Quién se convirtió en el nuevo primer ministro del Reino Unido tras la victoria del Partido Laborista en julio de 2024?",
        "respuesta_esperada": "Keir Starmer."
    },
    {
        "id": 72,
        "tipo": "Factual",
        "pregunta": "¿Qué nombre reciben las esperadas gafas de realidad mixta lanzadas por la empresa Apple?",
        "respuesta_esperada": "Apple Vision Pro."
    },
    {
        "id": 73,
        "tipo": "Factual",
        "pregunta": "¿Cuál fue el precio oficial de salida al mercado de las gafas Apple Vision Pro en dólares?",
        "respuesta_esperada": "3.499 dólares."
    },
    {
        "id": 74,
        "tipo": "Inferencia",
        "pregunta": "¿Qué caracteristica tiene el nuevo modelo de Apple Vision Pro para poder conectarlas a los Mac?",
        "respuesta_esperada": "Una conexión física mediante cable."
    },
    {
        "id": 75,
        "tipo": "Factual",
        "pregunta": "¿En qué mes y año lanzó Apple al mercado estadounidense su primer dispositivo de gafas Vision Pro?",
        "respuesta_esperada": "En febrero de 2024."
    },
    {
        "id": 76,
        "tipo": "Factual",
        "pregunta": "¿Qué empresa tecnológica es la creadora y fabricante de las populares gafas de realidad mixta llamadas Vision Pro?",
        "respuesta_esperada": "Apple."
    },
    {
        "id": 77,
        "tipo": "Factual",
        "pregunta": "¿En qué fecha exacta se produjo el gran eclipse solar total que oscureció Norteamérica en 2024?",
        "respuesta_esperada": "El 8 de abril de 2024."
    },
    {
        "id": 78,
        "tipo": "Factual",
        "pregunta": "¿En qué tres países del continente americano se pudo observar la fase de totalidad del eclipse solar de abril de 2024?",
        "respuesta_esperada": "México, Estados Unidos y Canadá."
    },
    {
        "id": 79,
        "tipo": "Factual",
        "pregunta": "Según las noticias de abril de 2024, ¿qué fenómeno astronómico oscureció parte de Norteamérica atrayendo el gran interés de la ciencia?",
        "respuesta_esperada": "Un eclipse solar total."
    },
    {
        "id": 80,
        "tipo": "Inferencia",
        "pregunta": "¿En qué región concreta del continente americano se pudo aprovechar científicamente y de forma óptima el gran eclipse solar del 8 de abril de 2024?",
        "respuesta_esperada": "En América del Norte (Norteamérica)."
    },
    {
        "id": 81,
        "tipo": "Factual",
        "pregunta": "¿En qué fecha exacta se lanzó sin tripulación la misión Artemis I de la NASA?",
        "respuesta_esperada": "El 16 de noviembre de 2022."
    },
    {
        "id": 82,
        "tipo": "Factual",
        "pregunta": "¿Para qué mes y año está previsto ahora el lanzamiento de la misión Artemis II tras su aplazamiento?",
        "respuesta_esperada": "Para abril de 2026."
    },
    {
        "id": 83,
        "tipo": "Factual",
        "pregunta": "¿Qué modelo de trajes espaciales usarán los astronautas de la misión Artemis III en la superficie lunar?",
        "respuesta_esperada": "Los trajes AxEMU."
    },
    {
        "id": 84,
        "tipo": "Factual",
        "pregunta": "¿En qué estado de Estados Unidos se encuentra el Centro Marshall donde los ingenieros de la NASA han realizado simulaciones acústicas?",
        "respuesta_esperada": "En Alabama."
    },
    {
        "id": 85,
        "tipo": "Factual",
        "pregunta": "Según la NASA, ¿dónde caerá exactamente la nave Orión al regresar a la Tierra tras la misión Artemis II?",
        "respuesta_esperada": "En el Pacífico oriental, cerca de la costa de la ciudad de San Diego."
    },
    {
        "id": 86,
        "tipo": "Factual",
        "pregunta": "Según la consultora Drewry, ¿qué porcentaje exacto se elevó el flete de transporte marítimo entre Shanghái y Róterdam de enero a junio de 2024?",
        "respuesta_esperada": "Un 233%."
    },
    {
        "id": 87,
        "tipo": "Inferencia",
        "pregunta": "Debido al desvío de barcos por el Cabo de Buena Esperanza, ¿cuántas millas náuticas se incrementa el trayecto entre Shanghái y Génova?",
        "respuesta_esperada": "4.878 millas."
    },
    {
        "id": 88,
        "tipo": "Factual",
        "pregunta": "¿Cuál fue la única de las 10 principales navieras mundiales que registró pérdidas durante el inicio de la crisis del Mar Rojo en 2024?",
        "respuesta_esperada": "Maersk."
    },
    {
        "id": 89,
        "tipo": "Factual",
        "pregunta": "De acuerdo con el Banco de España, ¿cuál fue el porcentaje de retroceso de las ventas minoristas en la zona euro durante el año 2023?",
        "respuesta_esperada": "Un 1% (un punto porcentual)."
    },
    {
        "id": 90,
        "tipo": "Factual",
        "pregunta": "¿Cuánto tiempo duró el atasco provocado por el megabuque Ever Given en el Canal de Suez en el año 2021?",
        "respuesta_esperada": "Seis días."
    },
    {
        "id": 91,
        "tipo": "Factual",
        "pregunta": "¿Cuál fue el índice de precios al consumidor (IPC) exacto registrado en Argentina durante el mes de diciembre de 2023?",
        "respuesta_esperada": "Un 25,5%."
    },
    {
        "id": 92,
        "tipo": "Factual",
        "pregunta": "Según el INDEC, ¿cuál fue la cifra oficial de inflación en Argentina correspondiente al mes de diciembre de 2024?",
        "respuesta_esperada": "Un 2,7%."
    },
    {
        "id": 93,
        "tipo": "Factual",
        "pregunta": "¿A qué porcentaje de la población argentina alcanzó la pobreza durante el primer semestre de 2024?",
        "respuesta_esperada": "Al 52,9%."
    },
    {
        "id": 94,
        "tipo": "Factual",
        "pregunta": "Según la política económica de Javier Milei, ¿a qué porcentaje mensual se mantuvo el ritmo de devaluación del peso durante 2024 para frenar la inflación?",
        "respuesta_esperada": "A un ritmo del 2% mensual."
    },
    {
        "id": 95,
        "tipo": "Factual",
        "pregunta": "¿A cuánto asciende la indemnización (en dólares) que la jueza Loretta Preska sentenció que Argentina debe pagar por la nacionalización de YPF?",
        "respuesta_esperada": "16.099 millones de dólares."
    },
    {
        "id": 96,
        "tipo": "Factual",
        "pregunta": "¿Cuál es el nombre del gas inerte y del elemento radiactivo de vida corta que se producen como residuos en el proceso de fusión nuclear?",
        "respuesta_esperada": "Helio y tritio."
    },
    {
        "id": 97,
        "tipo": "Factual",
        "pregunta": "Según los datos del Laboratorio Nacional Lawrence Livermore de 2022, ¿cuál fue la producción de energía obtenida mediante fusión nuclear medida en megajulios (MJ)?",
        "respuesta_esperada": "Superior a 3,5 MJ."
    },
    {
        "id": 98,
        "tipo": "Factual",
        "pregunta": "¿Qué organismo o laboratorio logró por primera vez generar más energía mediante fusión nuclear de la necesaria para llevarla a cabo?",
        "respuesta_esperada": "El Laboratorio Nacional Lawrence Livermore (LLNL)."
    },
    {
        "id": 99,
        "tipo": "Factual",
        "pregunta": "Según la Agencia Internacional de la Energía (AIE), ¿cuál será la producción eléctrica generada por reactores nucleares en el mundo durante 2025?",
        "respuesta_esperada": "2.900 TWh."
    },
    {
        "id": 100,
        "tipo": "Trampa",
        "pregunta": "¿Qué porcentaje del suministro mundial de uranio enriquecido procede de Kazajistán, según los datos de la AIE en 2025?",
        "respuesta_esperada": "Ninguno, Kazajistán extrae el uranio (43%), pero el uranio enriquecido procede de Rusia (40%)."
    },
    {
        "id": 101,
        "tipo": "Factual",
        "pregunta": "¿Qué dos compuestos químicos detectó el telescopio James Webb en la atmósfera del exoplaneta K2-18 b?",
        "respuesta_esperada": "Dimetil sulfuro (DMS) y dimetil disulfuro (DMDS)."
    },
    {
        "id": 102,
        "tipo": "Factual",
        "pregunta": "¿Cómo se ha bautizado a la cadena masiva de al menos 20 galaxias descubierta por el telescopio James Webb?",
        "respuesta_esperada": "Cosmic vine."
    },
    {
        "id": 103,
        "tipo": "Inferencia",
        "pregunta": "Según el tamaño de la Vía Láctea detallado en la noticia, ¿cuántas veces es más ancha la estructura 'Cosmic vine' en comparación a nuestra galaxia?",
        "respuesta_esperada": "Unas 6,5 veces más ancha (la Vía Láctea tiene 100.000 años luz de ancho y Cosmic vine 650.000)."
    },
    {
        "id": 104,
        "tipo": "Factual",
        "pregunta": "¿Cuál era el diámetro del espejo primario del Telescopio Espacial Hubble en comparación con los 6.5 metros del James Webb?",
        "respuesta_esperada": "2,4 metros."
    },
    {
        "id": 105,
        "tipo": "Trampa",
        "pregunta": "¿En qué año lanzó el científico James Webb personalmente la primera sonda interplanetaria a Marte?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 106,
        "tipo": "Factual",
        "pregunta": "¿Qué observatorio espacial detectó por primera vez el potente resplandor de rayos gamma GRB 250314A en marzo?",
        "respuesta_esperada": "El observatorio espacial SVOM (un proyecto conjunto entre Francia y China)."
    },
    {
        "id": 107,
        "tipo": "Factual",
        "pregunta": "¿Cuántos años después del Big Bang ocurrió aproximadamente la explosión de la supernova captada por el Telescopio Muy Grande (VLT)?",
        "respuesta_esperada": "Aproximadamente 730 millones de años después del Big Bang."
    },
    {
        "id": 108,
        "tipo": "Factual",
        "pregunta": "En su discurso desde el balcón del Palacio Nacional, ¿qué porcentaje exacto de los votos afirmó Nayib Bukele haber logrado en las elecciones presidenciales?",
        "respuesta_esperada": "Más del 85% de los votos."
    },
    {
        "id": 109,
        "tipo": "Factual",
        "pregunta": "Antes de fundar el movimiento Nuevas Ideas, ¿de qué partido político provocó Nayib Bukele su propia expulsión en el año 2017?",
        "respuesta_esperada": "Del Frente Farabundo Martí para la Liberación Nacional (FMLN)."
    },
    {
        "id": 110,
        "tipo": "Factual",
        "pregunta": "Según los datos de la Policía Nacional Civil (PNC), ¿cuántos días consecutivos sin asesinatos se registraban en El Salvador el día de la jornada electoral?",
        "respuesta_esperada": "El décimo séptimo (17) día consecutivo sin asesinatos."
    },
    {
        "id": 111,
        "tipo": "Factual",
        "pregunta": "¿Cuál es la capacidad máxima de reclusos que tiene el Centro de Confinamiento del Terrorismo creado en El Salvador?",
        "respuesta_esperada": "Tiene capacidad para 40.000 pandilleros."
    },
    {
        "id": 112,
        "tipo": "Trampa",
        "pregunta": "¿Qué presidente Iraní tildó de fraude electoral la reelección de Nayib Bukele?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 113,
        "tipo": "Inferencia",
        "pregunta": "Considerando las declaraciones de Katrin Deschauer, ¿reconoció Alemania oficialmente los resultados finales de las elecciones salvadoreñas el mismo lunes?",
        "respuesta_esperada": "No, Alemania fue cauta porque el conteo oficial del Tribunal Supremo Electoral aún no había terminado."
    },
    {
        "id": 114,
        "tipo": "Factual",
        "pregunta": "¿Para combatir qué dos enfermedades se aprobó en Reino Unido y Estados Unidos la primera terapia génica basada en el método CRISPR/Cas9?",
        "respuesta_esperada": "La anemia de células falciformes y la beta talasemia."
    },
    {
        "id": 115,
        "tipo": "Factual",
        "pregunta": "¿De qué parte específica del cuerpo humano se extraen habitualmente las células madre mediante una punción en la cresta ilíaca?",
        "respuesta_esperada": "De la médula ósea (en la cadera)."
    },
    {
        "id": 116,
        "tipo": "Factual",
        "pregunta": "¿Qué dos científicos recibieron el Premio Nobel por sus descubrimientos en torno a la reprogramación celular?",
        "respuesta_esperada": "Yamanaka y Guston."
    },
    {
        "id": 117,
        "tipo": "Factual",
        "pregunta": "Según las recomendaciones dermatológicas citadas en la noticia, ¿qué factor de protección solar mínimo (SPF) deben utilizar las personas de piel clara?",
        "respuesta_esperada": "Factor 50 SPF."
    },
    {
        "id": 118,
        "tipo": "Inferencia",
        "pregunta": "Si a un paciente le clasifican un carcinoma utilizando el sistema TNM y se centran en evaluar la letra 'M', ¿qué están buscando médicamente?",
        "respuesta_esperada": "La existencia o no de metástasis."
    },
    {
        "id": 119,
        "tipo": "Factual",
        "pregunta": "Según el descubrimiento de dos tipos de células madre responsables de formar las raíces dentales, ¿qué tipo de piezas bucales podrían hacer crecer los nuevos tratamientos?",
        "respuesta_esperada": "Dientes naturales."
    },
    {
        "id": 120,
        "tipo": "Factual",
        "pregunta": "¿Cuántos metros de diámetro tendrá el túnel submarino proyectado para conectar Génova Oeste con Génova Este?",
        "respuesta_esperada": "16 metros de diámetro."
    },
    {
        "id": 121,
        "tipo": "Factual",
        "pregunta": "¿En qué año está previsto que finalice la construcción del gran túnel submarino de Génova financiado por Autostrade per l'Italia?",
        "respuesta_esperada": "En el año 2029."
    },
    {
        "id": 122,
        "tipo": "Factual",
        "pregunta": "¿Qué ciudad marroquí y qué ciudad española conectaría el tren que circulará por el futuro túnel submarino del Estrecho de Gibraltar?",
        "respuesta_esperada": "Casablanca y Madrid."
    },
    {
        "id": 123,
        "tipo": "Factual",
        "pregunta": "¿Cómo se llama la fragata que Alemania envió a Copenhague para proteger la doble cumbre de la UE frente a la amenaza de drones?",
        "respuesta_esperada": "La fragata Hamburg."
    },
    {
        "id": 124,
        "tipo": "Inferencia",
        "pregunta": "Al ser avistados submarinos rusos en el Mar Báltico, ¿por qué los barcos de la defensa danesa justifican seguirlos como una práctica 'normal'?",
        "respuesta_esperada": "Se hace con fines disuasorios y de vigilancia porque las embarcaciones rusas no pertenecen a la OTAN."
    },
    {
        "id": 125,
        "tipo": "Trampa",
        "pregunta": "¿Qué carguero ruso dañó presuntamente dos cables submarinos de telecomunicaciones en el Báltico en noviembre de 2024?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 126,
        "tipo": "Factual",
        "pregunta": "En la cumbre COP29, ¿qué cantidad de dinero se aprobó movilizar para el año 2035 destinada a países en desarrollo?",
        "respuesta_esperada": "300.000 millones de dólares."
    },
    {
        "id": 127,
        "tipo": "Factual",
        "pregunta": "Según las estimaciones de los expertos mencionadas en la cumbre de Brasil, ¿a qué cifra ascienden las verdaderas necesidades financieras climáticas anuales exigidas para 2035?",
        "respuesta_esperada": "A 1,3 billones de dólares."
    },
    {
        "id": 128,
        "tipo": "Factual",
        "pregunta": "¿Cómo se llama el mecanismo de la Unión Europea que impondrá un cargo a las importaciones de bienes con altas emisiones de carbono a partir del 1 de enero de 2016?",
        "respuesta_esperada": "Mecanismo de Ajuste en Frontera por Carbono."
    },
    {
        "id": 129,
        "tipo": "Inferencia",
        "pregunta": "Teniendo en cuenta la postura de China en la cumbre, ¿se mostró este país abierto a debatir el acortamiento de los plazos para presentar sus planes de acción climática (NDC)?",
        "respuesta_esperada": "No, China se ha mostrado firme en que no debería haber debate sobre estos planes."
    },
    {
        "id": 130,
        "tipo": "Factual",
        "pregunta": "¿En qué ciudad de la Amazonia brasileña se celebró la conferencia de la ONU sobre el clima conocida como COP30?",
        "respuesta_esperada": "En la ciudad de Belém."
    },
    {
        "id": 131,
        "tipo": "Trampa",
        "pregunta": "¿Qué nuevos compromisos de financiación aportó la delegación de Estados Unidos durante su intervención en la cumbre de Belém?",
        "respuesta_esperada": "Ninguno, ya que Estados Unidos, a pesar de ser el mayor emisor histórico, estuvo ausente en la cumbre de Belém."
    },
    {
        "id": 132,
        "tipo": "Factual",
        "pregunta": "Según un nuevo análisis publicado por la Organización Mundial de la Salud (OMS), ¿cuál es la relación probada entre las vacunas y el autismo?",
        "respuesta_esperada": "Se confirma que no existe ninguna relación entre las vacunas y el autismo."
    },
    {
        "id": 133,
        "tipo": "Factual",
        "pregunta": "¿Qué compañía farmacéutica japonesa desarrolló la primera vacuna contra la viruela del mono aprobada por la OMS para niños?",
        "respuesta_esperada": "KM Biologics."
    },
    {
        "id": 134,
        "tipo": "Factual",
        "pregunta": "¿A partir de qué edad puede utilizarse la vacuna contra la viruela del mono fabricada por KM Biologics?",
        "respuesta_esperada": "A partir del primer año de edad."
    },
    {
        "id": 135,
        "tipo": "Factual",
        "pregunta": "¿Cómo se llama la alianza que facilita las inmunizaciones en países en desarrollo y que precisa la aprobación de la OMS para llevar la vacuna a la República Democrática del Congo?",
        "respuesta_esperada": "GAVI."
    },
    {
        "id": 136,
        "tipo": "Factual",
        "pregunta": "¿Cuántos pacientes espera tener implantados el Instituto Chino de Investigación Cerebral (CIBR) con el chip Beinao No. 1 para finales de año?",
        "respuesta_esperada": "Trece personas."
    },
    {
        "id": 137,
        "tipo": "Factual",
        "pregunta": "¿Cuál es la empresa tecnológica que ha desarrollado el chip visual Prima, de dos milímetros cuadrados, plantando cara a Neuralink?",
        "respuesta_esperada": "Science Corporation."
    },
    {
        "id": 138,
        "tipo": "Factual",
        "pregunta": "Según la historia estratégica detallada en el artículo, ¿en qué año fundó Elon Musk la compañía Neuralink?",
        "respuesta_esperada": "En 2026."
    },
    {
        "id": 139,
        "tipo": "Factual",
        "pregunta": "¿Cuál es el peso exacto del implante biotecnológico desarrollado por Neuralink?",
        "respuesta_esperada": "Cuatro gramos."
    },
    {
        "id": 140,
        "tipo": "Factual",
        "pregunta": "¿Cuáles son las previsiones de facturación y número de pacientes implantados por curso que Neuralink espera alcanzar en el año 2031?",
        "respuesta_esperada": "1.000 millones de facturación y 20.000 pacientes."
    },
    # Fuente: Elon Musk vuelve a estar en problemas: falla el primer y polémico implante cerebral de Neuralink
    {
        "id": 141,
        "tipo": "Factual",
        "pregunta": "¿Cómo se llama el primer paciente humano que recibió el implante cerebral de Neuralink y que quedó tetrapléjico en 2016?",
        "respuesta_esperada": "Noland Arbaugh."
    },
    # Fuente: Elon Musk vuelve a estar en problemas: falla el primer y polémico implante cerebral de Neuralink
    {
        "id": 142,
        "tipo": "Factual",
        "pregunta": "¿Qué significan las siglas del ensayo PRIME en el que participa el primer paciente de Neuralink?",
        "respuesta_esperada": "Precise Robotically Implanted Brain-Computer Interface."
    },
    # Fuente: Elon Musk vuelve a estar en problemas: falla el primer y polémico implante cerebral de Neuralink
    {
        "id": 143,
        "tipo": "Trampa",
        "pregunta": "¿En qué hospital exacto de la ciudad de Nueva York se le implantó el chip al paciente Noland Arbaugh?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: Colombia desafía a España por el galeón San José
    {
        "id": 144,
        "tipo": "Factual",
        "pregunta": "Según el artículo sobre el desafío de Colombia frente a España, ¿a finales de qué siglo se hundió el navío San José?",
        "respuesta_esperada": "A finales del siglo XVII."
    },
    # Fuente: Galeón San José: España propone a Colombia un acuerdo sobre patrimonio subacuático
    {
        "id": 145,
        "tipo": "Factual",
        "pregunta": "Según la historia del buque, ¿qué día exacto fue hundido el galeón San José por una flota de corsarios ingleses?",
        "respuesta_esperada": "El 8 de junio de 1708."
    },
    # Fuente: Galeón San José: España propone a Colombia un acuerdo sobre patrimonio subacuático
    {
        "id": 146,
        "tipo": "Factual",
        "pregunta": "¿Cuántas monedas de oro y plata recogidas en Portobelo (Panamá) llevaba aproximadamente el galeón en su interior?",
        "respuesta_esperada": "Cerca de 11 millones de monedas de ocho escudos en oro y plata."
    },
    # Fuente: Un cañón, porcelanas chinas y monedas son extraídas por Colombia del galeón San José
    {
        "id": 147,
        "tipo": "Factual",
        "pregunta": "¿Qué palabra concreta figura inscrita en el cañón de artillería extraído del galeón San José?",
        "respuesta_esperada": "La palabra \"Sevilla\"."
    },
    # Fuente: Crean un galeón San José de chocolate y su tesoro será repartido a niños
    {
        "id": 148,
        "tipo": "Factual",
        "pregunta": "¿Qué dos personas elaboraron una escultura de chocolate del galeón San José sostenido por un pulpo gigante?",
        "respuesta_esperada": "El chef pastelero Juan Arellano y Andrea Torres."
    },
    # Fuente: Galeón San José: España propone a Colombia un acuerdo sobre patrimonio subacuático
    {
        "id": 149,
        "tipo": "Trampa",
        "pregunta": "¿Cuál era el nombre del capitán corsario inglés que dio la orden de hundir personalmente el barco en 1708?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: Colombia desafía a España por el galeón San José
    {
        "id": 150,
        "tipo": "Inferencia",
        "pregunta": "¿Por qué motivo acudirá una delegación del pueblo Qhara Qhara como testigo a las operaciones submarinas en el galeón San José?",
        "respuesta_esperada": "Para asegurarse de que el gobierno colombiano no se apropie bienes de los pueblos originarios."
    },
    # Fuente: “Hay microplásticos en la lluvia, en nuestra sangre y en cerebros”
    {
        "id": 151,
        "tipo": "Factual",
        "pregunta": "¿A cuántos campos de fútbol equivale la superficie del gigantesco vertedero de Ghazipur situado en la India?",
        "respuesta_esperada": "A 35 campos de fútbol."
    },
    # Fuente: ¿Qué agua tiene más microplásticos en España? Esto es lo que dice un estudio
    {
        "id": 152,
        "tipo": "Factual",
        "pregunta": "¿Cuál fue la concentración media exacta de microplásticos (MP) por litro detectada en el agua embotellada de las cinco marcas españolas analizadas?",
        "respuesta_esperada": "0,7 MP por litro."
    },
    # Fuente: ¿Qué agua tiene más microplásticos en España? Esto es lo que dice un estudio
    {
        "id": 153,
        "tipo": "Factual",
        "pregunta": "¿En qué revista científica se publicaron los resultados de la segunda parte del estudio centrada en el agua embotellada?",
        "respuesta_esperada": "En la revista Scientific Reports."
    },
    # Fuente: Encuentran microplásticos en penes humanos por primera vez
    {
        "id": 154,
        "tipo": "Factual",
        "pregunta": "Según las imágenes químicas del estudio peneano, ¿cuáles fueron los dos tipos de microplásticos más prevalentes encontrados en las muestras?",
        "respuesta_esperada": "El tereftalato de polietileno (PET) y el polipropileno (PP)."
    },
    # Fuente: Encuentran microplásticos en penes humanos por primera vez
    {
        "id": 155,
        "tipo": "Factual",
        "pregunta": "Según el estudio publicado en mayo por el toxicólogo Matthew J. Campen, ¿cuántos testículos de perro se analizaron para compararlos con los 23 de humanos?",
        "respuesta_esperada": "47 testículos de perro."
    },
    # Fuente: Encuentran microplásticos en penes humanos por primera vez
    {
        "id": 156,
        "tipo": "Trampa",
        "pregunta": "¿Qué marca específica de agua embotellada fue la que dio un resultado más alto en toxinas en la investigación del toxicólogo Campen?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: “Hay microplásticos en la lluvia, en nuestra sangre y en cerebros”
    {
        "id": 157,
        "tipo": "Trampa",
        "pregunta": "¿En qué año exacto ha aprobado California la ley que prohibirá enviar chatarra electrónica de segunda mano a países de África?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: ¿Qué agua tiene más microplásticos en España? Esto es lo que dice un estudio
    {
        "id": 158,
        "tipo": "Inferencia",
        "pregunta": "Basándonos en las cifras del estudio de Environplanet, ¿qué tipo de agua tiene una menor cantidad de partículas no plásticas de origen artificial (PANP) por litro: la de grifo o la embotellada?",
        "respuesta_esperada": "El agua de grifo (presenta 0,0322 PANP/litro frente a los 1,70 del agua embotellada)."
    },
    # Fuente: Meta no lanzará su inteligencia artificial más potente en la Unión Europea
    {
        "id": 159,
        "tipo": "Factual",
        "pregunta": "¿Cómo se llama la versión de inteligencia artificial de Meta que sí llegará a la Unión Europea porque solo es compatible con texto?",
        "respuesta_esperada": "Llama 3."
    },
    # Fuente: Meta lanza su asistente de inteligencia artificial generativa en la Unión Europea
    {
        "id": 160,
        "tipo": "Factual",
        "pregunta": "¿En cuántos idiomas estará disponible el asistente Meta AI en su versión europea enfocada únicamente en la generación de texto?",
        "respuesta_esperada": "En seis idiomas."
    },
    # Fuente: Meta lanza su asistente de inteligencia artificial generativa en la Unión Europea
    {
        "id": 161,
        "tipo": "Factual",
        "pregunta": "¿Cuánto dinero planea invertir la empresa Meta este año para hacer de la inteligencia artificial una prioridad estratégica?",
        "respuesta_esperada": "Entre 60.000 y 65.000 millones de dólares."
    },
    # Fuente: Meta no lanzará su inteligencia artificial más potente en la Unión Europea
    {
        "id": 162,
        "tipo": "Trampa",
        "pregunta": "¿Qué cuantía económica tiene la multa que la Comisión Europea impuso a Meta en mayo por el uso de algoritmos adictivos?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: Meta lanza su asistente de inteligencia artificial generativa en la Unión Europea
    {
        "id": 163,
        "tipo": "Inferencia",
        "pregunta": "Atendiendo a las características de la versión europea de Meta AI, ¿utilizó la empresa los datos de los internautas europeos para entrenar a su asistente virtual?",
        "respuesta_esperada": "No, Meta AI no se entrenó con datos de usuarios europeos."
    },
    # Fuente: Cambios bruscos en la temperatura de la Luna disparan las existencias de agua helada
    {
        "id": 164,
        "tipo": "Factual",
        "pregunta": "En el lugar de aterrizaje del módulo Chandrayaan-3, ¿qué temperatura máxima se alcanzó en una pendiente orientada hacia el Sol con un ángulo de 6 grados?",
        "respuesta_esperada": "Alcanzó un máximo de 82 grados."
    },
    # Fuente: Hallan indicios de un antiguo océano de magma en el polo sur de la Luna
    {
        "id": 165,
        "tipo": "Factual",
        "pregunta": "¿Con qué instrumento tecnológico realizó el vehículo Pragyan sus 23 mediciones a lo largo de 103 metros en la superficie lunar?",
        "respuesta_esperada": "Utilizando su espectrómetro de rayos X de partículas alfa."
    },
    # Fuente: La sonda ‘Athena’ cayó en un cráter y de lado: su misión en el Polo Sur de la Luna ha terminado
    {
        "id": 166,
        "tipo": "Factual",
        "pregunta": "¿A qué distancia exacta de su ubicación deseada confirmó la NASA que había aterrizado de manera accidentada la sonda Athena?",
        "respuesta_esperada": "A 400 metros de distancia."
    },
    # Fuente: No solo Estados Unidos o Rusia: te sorprenderá saber la lista de países que han llegado alguna vez a la Luna
    {
        "id": 167,
        "tipo": "Factual",
        "pregunta": "¿En qué fecha exacta logró Japón el alunizaje exitoso de su nave Smart Lander con su módulo inteligente SLIM?",
        "respuesta_esperada": "El 20 de enero de 2024."
    },
    # Fuente: La sonda ‘Athena’ cayó en un cráter y de lado: su misión en el Polo Sur de la Luna ha terminado
    {
        "id": 168,
        "tipo": "Trampa",
        "pregunta": "¿Cuál era el nombre del astronauta que controló de manera remota el aterrizaje de la sonda Athena desde el centro de control en Houston?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: No solo Estados Unidos o Rusia: te sorprenderá saber la lista de países que han llegado alguna vez a la Luna
    {
        "id": 169,
        "tipo": "Trampa",
        "pregunta": "¿Qué cantidad de kilos de roca lunar trajo a la Tierra la misión espacial china Chang'e 3 tras su alunizaje en el año 2013?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: La alta velocidad llega a las capitales gallegas y Asturias el martes tras tres años de retraso
    {
        "id": 170,
        "tipo": "Factual",
        "pregunta": "¿En qué fecha exacta entra en funcionamiento el AVE que completa el recorrido entre Madrid, las capitales gallegas y Asturias?",
        "respuesta_esperada": "A partir de este próximo martes, 21 de mayo."
    },
    # Fuente: La alta velocidad llega a las capitales gallegas y Asturias el martes tras tres años de retraso
    {
        "id": 171,
        "tipo": "Factual",
        "pregunta": "¿Qué cantidad económica en indemnizaciones reclama Renfe a la empresa Talgo debido a los tres años y medio de retraso en la entrega de los trenes S-106?",
        "respuesta_esperada": "166 millones de euros."
    },
    # Fuente: La alta velocidad llega a las capitales gallegas y Asturias el martes tras tres años de retraso
    {
        "id": 172,
        "tipo": "Inferencia",
        "pregunta": "Gracias a la puesta en marcha de los nuevos convoyes Avril, ¿cuántos minutos de viaje se ahorrarán los pasajeros que realicen el trayecto en alta velocidad entre Oviedo y Madrid?",
        "respuesta_esperada": "Se reducirá el tiempo de viaje en 12 minutos."
    },
    # Fuente: Aquellos trenes de alta velocidad
    {
        "id": 173,
        "tipo": "Trampa",
        "pregunta": "¿A qué velocidad máxima exacta circuló el tren con destino a Vigo mientras atravesaba los montes galaicos tras salir de Zamora?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: La alta velocidad llega a las capitales gallegas y Asturias el martes tras tres años de retraso
    {
        "id": 174,
        "tipo": "Factual",
        "pregunta": "¿Cuántos trenes totales se incluían en el contrato que Renfe adjudicó a Talgo por 1.281 millones de euros en el año 2016?",
        "respuesta_esperada": "30 trenes S106."
    },
    # Fuente: Chile saca músculo en el mercado privado del litio
    {
        "id": 175,
        "tipo": "Factual",
        "pregunta": "Según las declaraciones del ministro Mario Marcel, ¿cuántas empresas en total presentaron propuestas al Gobierno de Chile para explotar los 26 salares disponibles?",
        "respuesta_esperada": "54 empresas."
    },
    # Fuente: Chile saca músculo en el mercado privado del litio
    {
        "id": 176,
        "tipo": "Factual",
        "pregunta": "¿En qué mes y año comenzará a operar comercialmente la nueva sociedad creada por Codelco y SQM para explotar el litio en el salar de Atacama?",
        "respuesta_esperada": "En enero de 2025."
    },
    # Fuente: Chile saca músculo en el mercado privado del litio
    {
        "id": 177,
        "tipo": "Inferencia",
        "pregunta": "Para garantizar el control del Estado en la nueva asociación público-privada entre Codelco y SQM, ¿qué participación accionarial mantendrá la empresa estatal?",
        "respuesta_esperada": "Tendrá la mayoría de las acciones (el 50% más una)."
    },
    # Fuente: España reserva a lo grande su asalto al Mundial
    {
        "id": 178,
        "tipo": "Factual",
        "pregunta": "Durante el partido de clasificación disputado en Tiflis frente a Georgia, ¿qué jugador de la selección española anotó el primer gol de penalti en el minuto 11?",
        "respuesta_esperada": "Mikel Oyarzabal."
    },
    # Fuente: España reserva a lo grande su asalto al Mundial
    {
        "id": 179,
        "tipo": "Trampa",
        "pregunta": "¿Cuál fue el porcentaje exacto de posesión de balón que tuvo la selección española en la segunda parte del partido contra Georgia?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: Chile saca músculo en el mercado privado del litio
    {
        "id": 180,
        "tipo": "Trampa",
        "pregunta": "¿A qué precio exacto vendió la compañía SQM la tonelada de litio a China durante el último trimestre comercial de 2024?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: Con la Inteligencia Artificial se liga mejor (un empleo)
    {
        "id": 181,
        "tipo": "Factual",
        "pregunta": "¿En qué ciudad española tiene su sede la Agencia Española de Supervisión de la Inteligencia Artificial (Aesia)?",
        "respuesta_esperada": "En La Coruña."
    },
    # Fuente: Con la Inteligencia Artificial se liga mejor (un empleo)
    {
        "id": 182,
        "tipo": "Factual",
        "pregunta": "¿Cuántas entradas tiene la clasificación europea de competencias conocida como ESCO?",
        "respuesta_esperada": "Más de 13.000 entradas."
    },
    # Fuente: Con la Inteligencia Artificial se liga mejor (un empleo)
    {
        "id": 183,
        "tipo": "Inferencia",
        "pregunta": "Según la información aportada por la Xunta, ¿cuántos puestos de trabajo calificados como de 'urgente cobertura' hay en Galicia que no se logran cubrir?",
        "respuesta_esperada": "Unos 15.000 puestos."
    },
    # Fuente: Con la Inteligencia Artificial se liga mejor (un empleo)
    {
        "id": 184,
        "tipo": "Trampa",
        "pregunta": "¿Qué gran empresa tecnológica estadounidense ha desarrollado la herramienta de inteligencia artificial EMi para la Xunta de Galicia?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: El impacto de la inteligencia artificial en la industria de automóviles
    {
        "id": 185,
        "tipo": "Factual",
        "pregunta": "¿Qué empresa utiliza en California vehículos sin conductor para entregar suministros médicos?",
        "respuesta_esperada": "La empresa Nuro."
    },
    # Fuente: El impacto de la Inteligencia Artificial en el periodismo
    {
        "id": 186,
        "tipo": "Factual",
        "pregunta": "¿En qué ciudad de Colombia se celebra el VIII Congreso de Editores de Europa y América Latina?",
        "respuesta_esperada": "En Cartagena de Indias."
    },
    # Fuente: Un tratamiento ya prohibido generó casos de transmisión de alzhéimer entre personas
    {
        "id": 187,
        "tipo": "Factual",
        "pregunta": "¿Qué tratamiento médico específico procedente de cadáveres generó casos de transmisión de alzhéimer entre personas?",
        "respuesta_esperada": "Un tratamiento con hormona del crecimiento."
    },
    # Fuente: Estudian ALZ-801 como alternativa oral en el tratamiento del Alzheimer
    {
        "id": 188,
        "tipo": "Factual",
        "pregunta": "¿Cómo se llama el fármaco oral en fase de estudio que busca prevenir el Alzheimer bloqueando los oligómeros amiloides?",
        "respuesta_esperada": "ALZ-801."
    },
    # Fuente: Google acelera investigación del Alzheimer con su IA, AlphaFold
    {
        "id": 189,
        "tipo": "Factual",
        "pregunta": "¿Qué herramienta de inteligencia artificial utiliza Google para acelerar el estudio de proteínas relacionadas con el Alzheimer?",
        "respuesta_esperada": "AlphaFold."
    },
    # Fuente: Un poema como acto de memoria ante el Alzhéimer
    {
        "id": 190,
        "tipo": "Factual",
        "pregunta": "¿Quién es el autor colombiano del libro de poesía 'Grietas de la luz' (2024) que honra la memoria de sus abuelas?",
        "respuesta_esperada": "Federico Díaz Granados."
    },
    # Fuente: Google acelera investigación del Alzheimer con su IA, AlphaFold
    {
        "id": 191,
        "tipo": "Trampa",
        "pregunta": "¿En qué año exacto descubrió Google la cura definitiva para el Alzheimer con su IA AlphaFold?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: El turista que visita el Maresme gasta más y dispara la recaudación de la tasa turística
    {
        "id": 192,
        "tipo": "Factual",
        "pregunta": "¿Cuál fue la cifra exacta de recaudación de la tasa turística en la comarca del Maresme durante el año 2024?",
        "respuesta_esperada": "4.662.602 euros."
    },
    # Fuente: El turista que visita el Maresme gasta más y dispara la recaudación de la tasa turística
    {
        "id": 193,
        "tipo": "Factual",
        "pregunta": "Según el informe del Maresme, ¿cuál es el porcentaje de hombres que integran el perfil del turista en la comarca?",
        "respuesta_esperada": "El 64,1%."
    },
    # Fuente: El turismo internacional en España bate récords: más de 84 millones de turistas y un gasto que supera los 108.
    {
        "id": 194,
        "tipo": "Factual",
        "pregunta": "Según las previsiones del Ministerio de Industria y Turismo en 2023, ¿cuántos millones de euros de gasto generaron los turistas internacionales en España?",
        "respuesta_esperada": "Más de 108.000 millones de euros."
    },
    # Fuente: El turismo espera una fuga de turistas de Canadá y México desde EE.UU. hacia España por los aranceles
    {
        "id": 195,
        "tipo": "Inferencia",
        "pregunta": "De acuerdo con los datos de Mesa del Turismo, ¿cuántos turistas mexicanos aproximadamente visitaron España el año pasado?",
        "respuesta_esperada": "Más de un millón de mexicanos."
    },
    # Fuente: El turismo espera una fuga de turistas de Canadá y México desde EE.UU. hacia España por los aranceles
    {
        "id": 196,
        "tipo": "Trampa",
        "pregunta": "¿Qué porcentaje exacto de impuestos aplicará Donald Trump a los turistas españoles que viajen a Nueva York?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: Las ventas de Tesla en España caen un 31% en octubre con un mercado de coches eléctricos al alza
    {
        "id": 197,
        "tipo": "Factual",
        "pregunta": "¿Qué porcentaje cayeron las ventas de la marca Tesla en España durante el mes de octubre?",
        "respuesta_esperada": "Un 31%."
    },
    # Fuente: ¿Quién fabrica las baterías de los coches eléctricos?
    {
        "id": 198,
        "tipo": "Factual",
        "pregunta": "¿Cuáles son los nombres de los dos gigantes chinos que concentran más del 50% del suministro global de baterías para coches eléctricos?",
        "respuesta_esperada": "CATL y BYD."
    },
    # Fuente: ¿Quién fabrica las baterías de los coches eléctricos?
    {
        "id": 199,
        "tipo": "Inferencia",
        "pregunta": "¿Cómo se denomina comercialmente la batería de producción propia que fabrica la empresa automovilística BYD a través de su filial FinDreams Battery?",
        "respuesta_esperada": "Blade Battery."
    },
    # Fuente: Definitivamente, la batería de los coches eléctricos durará más que los propios coches
    {
        "id": 200,
        "tipo": "Factual",
        "pregunta": "Según un nuevo estudio, ¿a qué porcentaje anual ha bajado la tasa de degradación de las baterías de coches eléctricos en los últimos cinco años?",
        "respuesta_esperada": "Ha bajado al 1,8% anual."
    },
    # Fuente: ¿Quién fabrica las baterías de los coches eléctricos?
    {
        "id": 201,
        "tipo": "Trampa",
        "pregunta": "¿Cuántas gigafactorías de baterías tiene actualmente operativas la marca BYD en territorio español?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: La agricultura ecológica pone en valor su triple impacto económico, social y ambiental
    {
        "id": 202,
        "tipo": "Factual",
        "pregunta": "¿Cuántas hectáreas de olivar dedica la Cooperativa Olivarera de Los Pedroches (Olipe) al cultivo biodinámico?",
        "respuesta_esperada": "300 hectáreas."
    },
    # Fuente: La agricultura ecológica pone en valor su triple impacto económico, social y ambiental
    {
        "id": 203,
        "tipo": "Inferencia",
        "pregunta": "De acuerdo con la reforma presupuestaria que se debate en las instituciones comunitarias, ¿qué porcentaje de reducción de fondos podrían sufrir las ayudas de la Política Agraria Común (PAC)?",
        "respuesta_esperada": "Una reducción del 20%."
    },
    # Fuente: Los agricultores firman un pacto con Agricultura para desconvocar las protestas
    {
        "id": 204,
        "tipo": "Factual",
        "pregunta": "¿Cuántos puntos recogía el pacto que firmaron los agricultores de la plataforma Gremi de la Pagesia con el departament d’Agricultura de Cataluña?",
        "respuesta_esperada": "19 puntos."
    },
    # Fuente: Los agricultores firman un pacto con Agricultura para desconvocar las protestas
    {
        "id": 205,
        "tipo": "Factual",
        "pregunta": "¿Cuántos años llevaba en su cargo el coordinador de Unió de Pagesos, Joan Caball, antes de certificarse su relevo?",
        "respuesta_esperada": "25 años."
    },
    # Fuente: Las consecuencias económicas de Mr. Trump
    {
        "id": 206,
        "tipo": "Factual",
        "pregunta": "¿Quién es el autor del libro 'El juego del dinero' citado en el artículo sobre la riqueza y la clase media?",
        "respuesta_esperada": "Gary Stevenson."
    },
    # Fuente: Los agricultores firman un pacto con Agricultura para desconvocar las protestas
    {
        "id": 207,
        "tipo": "Trampa",
        "pregunta": "¿Qué plaga específica de insectos devoradores de cultivos acordaron combatir en el pacto de 19 puntos firmado en Cataluña?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: Las empresas suspenden en ciberseguridad
    {
        "id": 208,
        "tipo": "Factual",
        "pregunta": "Según el informe de ISACA en 2024, ¿qué porcentaje de profesionales de ciberseguridad afirma que no cuentan con el presupuesto necesario?",
        "respuesta_esperada": "El 52%."
    },
    # Fuente: Las empresas suspenden en ciberseguridad
    {
        "id": 209,
        "tipo": "Factual",
        "pregunta": "¿Qué cantidad económica va a inyectar Google.org a la Universidad de Málaga para su programa de seminarios de ciberseguridad?",
        "respuesta_esperada": "Un millón de dólares."
    },
    # Fuente: La IA facilita y abarata los ataques informáticos
    {
        "id": 210,
        "tipo": "Factual",
        "pregunta": "¿En qué año fundó el empresario israelí Nir Zuk la firma de ciberseguridad Palo Alto Networks?",
        "respuesta_esperada": "En el año 2005."
    },
    # Fuente: La ciberseguridad preocupa a las pymes: el 60% han sido víctimas de ataques
    {
        "id": 211,
        "tipo": "Factual",
        "pregunta": "Según el informe del Observatorio de Digitalización de GoDaddy 2023, ¿qué porcentaje de pymes españolas han sido víctimas de algún ciberataque?",
        "respuesta_esperada": "El 60% de las empresas españolas."
    },
    # Fuente: La ciberseguridad preocupa a las pymes: el 60% han sido víctimas de ataques
    {
        "id": 212,
        "tipo": "Inferencia",
        "pregunta": "¿Para qué sirve específicamente la solución tecnológica HP Sure Recover mencionada en el informe de protección para pymes?",
        "respuesta_esperada": "Para ayudar a restaurar rápidamente la funcionalidad del sitio web en caso de ataques o caídas."
    },
    # Fuente: El Gobierno contempla que una de las empresas del sector sufriera un ataque informático
    {
        "id": 213,
        "tipo": "Trampa",
        "pregunta": "¿Qué grupo de hackers ruso reivindicó el último ataque a la infraestructura de Red Eléctrica?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    # Fuente: Los siete factores del sueño que impactan en la salud cardiovascular
    {
        "id": 214,
        "tipo": "Factual",
        "pregunta": "Según el estudio de la Asociación Estadounidense del Corazón, ¿cuántas horas de sueño por noche necesita la mayoría de los adultos?",
        "respuesta_esperada": "Entre 7 y 9 horas."
    },
    # Fuente: Así puede influir la dieta mediterránea en la salud mental
    {
        "id": 215,
        "tipo": "Factual",
        "pregunta": "En el estudio de la Universidad Tecnológica de Sidney sobre la dieta mediterránea, ¿cuántos hombres participaron y de qué rango de edad?",
        "respuesta_esperada": "Participaron 71 hombres de edades comprendidas entre 18 y 25 años."
    },
    # Fuente: Así puede influir la dieta mediterránea en la salud mental
    {
        "id": 216,
        "tipo": "Inferencia",
        "pregunta": "Según los investigadores, ¿a través de qué vía o nervio específico se comunican las bacterias del tracto intestinal con el cerebro?",
        "respuesta_esperada": "A través del nervio vago."
    },
    # Fuente: La dieta mediterránea protege la materia blanca del cerebro, según un nuevo estudio
    {
        "id": 217,
        "tipo": "Factual",
        "pregunta": "¿Cuántos participantes se incluyeron en el análisis de datos del estudio SOL-INCA-MRI enfocado en la materia blanca cerebral?",
        "respuesta_esperada": "Más de 2.774 participantes."
    },
    # Fuente: La dieta mediterránea protege la materia blanca del cerebro, según un nuevo estudio
    {
        "id": 218,
        "tipo": "Inferencia",
        "pregunta": "De todos los componentes de la dieta mediterránea, ¿qué dos alimentos específicos se asociaban de manera más estrecha con las características deseables de la materia blanca cerebral?",
        "respuesta_esperada": "El pescado y los cereales integrales."
    },
    # Fuente: Los siete factores del sueño que impactan en la salud cardiovascular
    {
        "id": 219,
        "tipo": "Factual",
        "pregunta": "¿En qué revista científica se publicó la nueva declaración de la Asociación Estadounidense del Corazón sobre el sueño y el cardiometabolismo?",
        "respuesta_esperada": "En la revista 'Circulation: Cardiovascular Quality and Outcomes'."
    },
    # Fuente: Así puede influir la dieta mediterránea en la salud mental
    {
        "id": 220,
        "tipo": "Trampa",
        "pregunta": "¿En qué hospital español se descubrió que la dieta mediterránea cura clínicamente la depresión profunda en dos semanas?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    }
    
]

# Convertimos a DataFrame
df_bateria = pd.DataFrame(datos)

# Mostramos una vista previa
print(f"✅ Batería creada con {len(df_bateria)} preguntas.")
display(df_bateria)

# Guardamos la batería en un archivo CSV para su uso posterior
df_bateria.to_csv("bateria_preguntas_tfg.csv", index=False, sep = ";", encoding= "utf-8-sig")
print("✅ Batería guardada como 'bateria_preguntas_tfg.csv'.")

✅ Batería creada con 220 preguntas.


,id,tipo,pregunta,respuesta_esperada
0,1,Factual,¿A cuanto llego a subir Trump los aranceles a ...,145%.
1,2,Factual,¿Cuánto dinero perdió Elon Musk tras el anunci...,17.800 millones de dólares.
2,3,Trampa,¿Qué opinó el Rey de España sobre los arancele...,No tengo información suficiente en mis archivos.
3,4,Trampa,¿Qué declaraciones hizo Elon Musk sobre los ar...,No tengo información suficiente en mis archivos.
4,5,Inferencia,¿Qué relación quiere Elon Musk entre Estados U...,Quiere una zona de libre comercio con arancele...
...,...,...,...,...
215,216,Inferencia,"Según los investigadores, ¿a través de qué vía...",A través del nervio vago.
216,217,Factual,¿Cuántos participantes se incluyeron en el aná...,Más de 2.774 participantes.
217,218,Inferencia,De todos los componentes de la dieta mediterrá...,El pescado y los cereales integrales.
218,219,Factual,¿En qué revista científica se publicó la nueva...,En la revista 'Circulation: Cardiovascular Qua...


✅ Batería guardada como 'bateria_preguntas_tfg.csv'.
